### 16. LOT 끝의 공통 움직임을 확인하고 `v5` 만들기

- `v3`의 센서값과 행은 그대로 두고, 확인한 센서의 후보만 해제함.
- `0`은 이 기준의 후보가 아니라는 뜻임. 설비 정상 확정은 아님.
- 기록 끝을 보고 정한 사후 검토임. 새로운 LOT의 실시간 판정 규칙은 아님.

### 1. 읽을 파일과 저장할 위치

- `cal01_base_v3.csv`와 함께 만든 `metadata/v3/dataset.json`이 필요함.
- 조업이벤트와 정비이력은 `data/` 또는 `data/raw/`에 둠.
- `SAVE_FILES=True`이면 `cal01_base_v5.csv`, 변경표, 그림을 저장함.
- 센서값 보간, 빈 초 생성, 학습용 표본 추출은 하지 않음.

In [ ]:
INPUT_FILE = "cal01_base_v3.csv"
OUTPUT_FILE = "cal01_base_v5.csv"
SAVE_FILES = True

import os
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
INPUT_CSV = os.path.join(PROCESSED_DIR, INPUT_FILE)
OUTPUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)
META_DIR = os.path.join(PROCESSED_DIR, "metadata", "v5")
REVIEW_DIR = os.path.join(PROCESSED_DIR, "review", "v5")
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "16_end_pattern_v5")
SOURCE_META = os.path.join(PROCESSED_DIR, "metadata", "v3", "dataset.json")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
ENC = "utf-8-sig"

# 직접 비교한 v3에만 아래 시간표를 적용함. 다른 CSV에는 자동으로 해제하지 않음.
REVIEWED_SHA256 = "247c6f328149427e93a588925a1e7b27a40cf3f9b3922997477d104d77094aa8"
with open(INPUT_CSV, "rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
with open(SOURCE_META, encoding="utf-8") as file:
    source_metadata = json.load(file)

review_applied = source_sha256 == REVIEWED_SHA256 and source_metadata.get("master_sha256") == source_sha256
# 경로를 바꾸더라도 v3를 덮어쓰지 않음.
write_allowed = SAVE_FILES and review_applied and os.path.realpath(INPUT_CSV) != os.path.realpath(OUTPUT_CSV)
if SAVE_FILES and not write_allowed:
    print("저장 보류: 검토한 v3와 생성 정보가 맞는지, 출력 경로가 입력과 다른지 확인하세요.")

# 문자열로 읽어 원본 값의 자릿수와 빈칸까지 유지함. 계산할 값만 따로 숫자로 바꿈.
base = pd.read_csv(INPUT_CSV, encoding=ENC, dtype=str, keep_default_na=False)
output = base.copy()
times = pd.to_datetime(base["MEAS_DT"])
not_excluded = base["EXCLUDED"].eq("False")
groups = source_metadata["groups"]
sensors = source_metadata["raw_sensor_columns"]
values = base[sensors].apply(pd.to_numeric, errors="coerce")
lots = sorted(base["LOT_NO"].unique())
lot_end = times.groupby(base["LOT_NO"]).max()
lot_start = times.groupby(base["LOT_NO"]).min()

# v3와 같은 LOT별 기준. 기존 제외는 평균과 표준편차 계산에서도 뺌.
reference = values.loc[not_excluded]
lot_mean = reference.groupby(base.loc[not_excluded, "LOT_NO"]).mean()
lot_std = reference.groupby(base.loc[not_excluded, "LOT_NO"]).std(ddof=0)

events = pd.read_csv(EVENT_CSV, encoding=ENC)
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
maintenance = pd.read_csv(MAINT_CSV, encoding=ENC)
maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
maintenance["STRT_DT"] = pd.to_datetime(maintenance["STRT_DT"])
maintenance["END_DT"] = pd.to_datetime(maintenance["END_DT"])

font_names = []
for font in font_manager.fontManager.ttflist:
    font_names.append(font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

if write_allowed:
    for folder in [META_DIR, REVIEW_DIR, FIG_DIR]:
        os.makedirs(folder, exist_ok=True)
print("입력 행:", len(base), "| LOT:", len(lots), "| 검토표 적용:", review_applied)

### 2. 14개 LOT의 마지막 2시간을 같은 위치에서 보기

- 가로축 `0`은 LOT의 마지막 기록임. `-20`은 마지막 기록 20분 전임.
- 표의 날짜와 시각은 실제 기록 시각임. 고점은 마지막 35분의 1분 평균 중 가장 높은 값임.
- `1분 평균`은 그림을 읽기 위한 요약임. v5에는 원래 관측 행을 유지함.
- `058`, `114`는 온도와 출력이 낮아진 채 끝나므로 이번 해제에서 뺌.
- 나머지 12개 LOT는 건조 온도가 올라갔다 내려가고 출력이 낮아졌다 다시 올라가는 모습이 반복됨.
  - 일부 LOT는 온도가 완전히 회복하기 전에 기록이 끝남. 회복 완료나 정상 운전이 확인됐다는 뜻은 아님.
- `Z3`, `Z6`는 기존 v3 판단을 유지함. 마지막 2시간 전체를 0으로 바꾸지 않음.

In [ ]:
lot_review_rows = []
for lot in lots:
    index = base.index[base["LOT_NO"].eq(lot)]
    last_index = index[times.loc[index].ge(lot_end[lot] - pd.Timedelta(minutes=35))]
    minute_values = values.loc[last_index].copy()
    minute_values.index = pd.DatetimeIndex(times.loc[last_index])
    minute_values = minute_values.resample("1min").mean()

    row = {
        "LOT": lot,
        "강종": base.loc[index[0], "GRADE"],
        "마지막 기록": lot_end[lot],
        "이번 처리": "건조 구역의 확인 범위만 해제",
    }
    for sensor in ["TC-Z1", "TC-Z2"]:
        peak = minute_values[sensor].idxmax()
        row[sensor + " 고점 시각"] = peak
        row[sensor + " 고점 ℃"] = round(float(minute_values.loc[peak, sensor]), 2)
        row[sensor + " 고점 후 최저 ℃"] = round(float(minute_values.loc[peak:, sensor].min()), 2)
    if lot in ["240058", "240114"]:
        row["이번 처리"] = "온도와 출력 하강이 이어짐, 기존 판단 유지"
    lot_review_rows.append(row)

lot_review = pd.DataFrame(lot_review_rows)
display(lot_review)

# 센서별로 14개 LOT를 모두 보여줌. 15초보다 큰 공백은 선을 잇지 않음.
overview_sensors = ["TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2", "TC-Z3", "TC-Z6"]
for sensor in overview_sensors:
    fig, axes = plt.subplots(7, 2, figsize=(14, 18), sharex=True)
    for axis, lot in zip(axes.flat, lots):
        index = base.index[base["LOT_NO"].eq(lot)]
        index = index[times.loc[index].ge(lot_end[lot] - pd.Timedelta(minutes=120))]
        clock = times.loc[index]
        pieces = clock.diff().dt.total_seconds().gt(15).cumsum()
        color = "#b5643e" if lot in ["240058", "240114"] else "#3f718a"
        for piece in pieces.unique():
            part = index[pieces.eq(piece)]
            trace = pd.Series(values.loc[part, sensor].to_numpy(), index=pd.DatetimeIndex(times.loc[part]))
            average = trace.resample("1min").mean()
            relative_time = (average.index - lot_end[lot]).total_seconds() / 60
            axis.plot(relative_time, average, color=color, linewidth=1)
        axis.set_title(lot + " | " + base.loc[index[0], "GRADE"], fontsize=10)
        axis.set_xlim(-120, 0)
        axis.grid(alpha=0.2)
        axis.axvline(-35, color="#aaaaaa", linestyle=":", linewidth=0.7)
    for axis in axes[-1]:
        axis.set_xlabel("마지막 기록까지 남은 시간 (분, 0이 마지막 기록)")
    unit = "℃" if sensor.startswith("TC-") else "%"
    fig.suptitle(sensor + " (" + unit + ") | 마지막 2시간, 1분 평균", fontsize=15)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    fig.text(0.05, 0.01, "갈색 058·114는 이번 해제에서 제외. 점선 오른쪽은 건조 구역을 자세히 비교한 마지막 35분.", fontsize=10)
    if write_allowed:
        fig.savefig(os.path.join(FIG_DIR, "01_all_lots_" + sensor + ".png"), dpi=120)
    plt.show()
    plt.close(fig)

### 3. 후보를 해제할 센서와 시간을 적기

- 위 14개 LOT 그림과 마지막 35분의 원자료를 비교해 아래 범위를 정했음.
- 온도는 마지막 봉우리와 그 뒤 내려간 부분, 출력은 온도 상승에 함께 낮아진 부분만 포함함.
- `START`는 포함하고 `END`는 포함하지 않음.
- `high`는 평균보다 높은 후보, `low`는 평균보다 낮은 후보임.
  - 같은 시간에 반대 방향으로 튄 값은 해제하지 않음.
- 출력이 다시 높아질 때의 짧은 튐은 이번 해제에 넣지 않음.
  - `121 OP-Z1`의 회복 뒤 떨어져 있는 낮은 후보 3행도 유지함.
- 이 범위는 현재 v3를 보고 정한 것임. 다른 파일에 마지막 35분 규칙으로 자동 적용하지 않음.

In [ ]:
review_ranges = pd.DataFrame([
    ['240001', 'TC-Z1', '2024-01-02 14:44:33', '2024-01-02 14:49:17', 'high'],
    ['240001', 'TC-Z1', '2024-01-02 14:53:45', '2024-01-02 14:59:16', 'low'],
    ['240001', 'TC-Z2', '2024-01-02 14:45:11', '2024-01-02 14:52:15', 'high'],
    ['240001', 'TC-Z2', '2024-01-02 14:56:53', '2024-01-02 15:03:40', 'low'],
    ['240001', 'OP-Z1', '2024-01-02 14:43:14', '2024-01-02 14:51:08', 'low'],
    ['240001', 'OP-Z2', '2024-01-02 14:45:03', '2024-01-02 14:54:42', 'low'],
    ['240024', 'TC-Z1', '2024-03-08 06:09:15', '2024-03-08 06:16:07', 'high'],
    ['240024', 'TC-Z1', '2024-03-08 06:22:13', '2024-03-08 06:29:30', 'low'],
    ['240024', 'TC-Z2', '2024-03-08 06:10:28', '2024-03-08 06:19:11', 'high'],
    ['240024', 'TC-Z2', '2024-03-08 06:24:20', '2024-03-08 06:29:49', 'low'],
    ['240024', 'OP-Z1', '2024-03-08 06:08:28', '2024-03-08 06:19:51', 'low'],
    ['240024', 'OP-Z2', '2024-03-08 06:10:46', '2024-03-08 06:23:16', 'low'],
    ['240037', 'TC-Z1', '2024-04-14 09:09:14', '2024-04-14 09:15:34', 'high'],
    ['240037', 'TC-Z1', '2024-04-14 09:18:56', '2024-04-14 09:24:51', 'low'],
    ['240037', 'TC-Z2', '2024-04-14 09:10:33', '2024-04-14 09:18:25', 'high'],
    ['240037', 'TC-Z2', '2024-04-14 09:21:53', '2024-04-14 09:27:20', 'low'],
    ['240037', 'OP-Z1', '2024-04-14 09:08:50', '2024-04-14 09:17:28', 'low'],
    ['240037', 'OP-Z2', '2024-04-14 09:11:07', '2024-04-14 09:19:57', 'low'],
    ['240038', 'TC-Z1', '2024-04-16 13:26:01', '2024-04-16 13:27:36', 'high'],
    ['240038', 'TC-Z1', '2024-04-16 13:33:52', '2024-04-16 13:37:08', 'low'],
    ['240038', 'TC-Z2', '2024-04-16 13:36:58', '2024-04-16 13:39:56', 'low'],
    ['240038', 'OP-Z1', '2024-04-16 13:24:04', '2024-04-16 13:29:31', 'low'],
    ['240038', 'OP-Z2', '2024-04-16 13:27:48', '2024-04-16 13:27:49', 'low'],
    ['240044', 'OP-Z1', '2024-05-03 19:46:25', '2024-05-03 19:50:17', 'low'],
    ['240044', 'OP-Z2', '2024-05-03 19:48:41', '2024-05-03 19:52:12', 'low'],
    ['240061', 'TC-Z1', '2024-06-15 04:42:29', '2024-06-15 04:44:59', 'low'],
    ['240061', 'TC-Z2', '2024-06-15 04:35:53', '2024-06-15 04:37:05', 'high'],
    ['240061', 'TC-Z2', '2024-06-15 04:45:33', '2024-06-15 04:49:20', 'low'],
    ['240061', 'OP-Z1', '2024-06-15 04:31:12', '2024-06-15 04:37:48', 'low'],
    ['240061', 'OP-Z2', '2024-06-15 04:33:45', '2024-06-15 04:40:52', 'low'],
    ['240069', 'TC-Z1', '2024-07-08 06:09:42', '2024-07-08 06:15:28', 'high'],
    ['240069', 'TC-Z1', '2024-07-08 06:19:18', '2024-07-08 06:25:12', 'low'],
    ['240069', 'TC-Z2', '2024-07-08 06:11:10', '2024-07-08 06:18:00', 'high'],
    ['240069', 'TC-Z2', '2024-07-08 06:22:06', '2024-07-08 06:29:32', 'low'],
    ['240069', 'OP-Z1', '2024-07-08 06:08:17', '2024-07-08 06:17:44', 'low'],
    ['240069', 'OP-Z2', '2024-07-08 06:10:25', '2024-07-08 06:20:50', 'low'],
    ['240091', 'TC-Z1', '2024-09-09 08:24:57', '2024-09-09 08:32:09', 'high'],
    ['240091', 'TC-Z1', '2024-09-09 08:38:57', '2024-09-09 08:44:13', 'low'],
    ['240091', 'TC-Z2', '2024-09-09 08:26:11', '2024-09-09 08:36:18', 'high'],
    ['240091', 'TC-Z2', '2024-09-09 08:40:40', '2024-09-09 08:44:13', 'low'],
    ['240091', 'OP-Z1', '2024-09-09 08:24:12', '2024-09-09 08:37:29', 'low'],
    ['240091', 'OP-Z2', '2024-09-09 08:26:47', '2024-09-09 08:39:42', 'low'],
    ['240108', 'TC-Z1', '2024-10-18 11:31:28', '2024-10-18 11:36:20', 'high'],
    ['240108', 'TC-Z1', '2024-10-18 11:40:40', '2024-10-18 11:45:38', 'low'],
    ['240108', 'TC-Z2', '2024-10-18 11:32:58', '2024-10-18 11:38:23', 'high'],
    ['240108', 'TC-Z2', '2024-10-18 11:43:34', '2024-10-18 11:49:34', 'low'],
    ['240108', 'OP-Z1', '2024-10-18 11:30:05', '2024-10-18 11:37:46', 'low'],
    ['240108', 'OP-Z2', '2024-10-18 11:32:28', '2024-10-18 11:40:43', 'low'],
    ['240117', 'TC-Z1', '2024-11-13 14:14:40', '2024-11-13 14:20:14', 'high'],
    ['240117', 'TC-Z1', '2024-11-13 14:29:36', '2024-11-13 14:32:20', 'low'],
    ['240117', 'TC-Z2', '2024-11-13 14:15:38', '2024-11-13 14:23:42', 'high'],
    ['240117', 'TC-Z2', '2024-11-13 14:30:22', '2024-11-13 14:32:20', 'low'],
    ['240117', 'OP-Z1', '2024-11-13 14:14:26', '2024-11-13 14:23:11', 'low'],
    ['240117', 'OP-Z2', '2024-11-13 14:15:47', '2024-11-13 14:28:08', 'low'],
    ['240121', 'TC-Z1', '2024-11-24 17:57:10', '2024-11-24 18:03:40', 'high'],
    ['240121', 'TC-Z1', '2024-11-24 18:07:59', '2024-11-24 18:14:42', 'low'],
    ['240121', 'TC-Z2', '2024-11-24 17:58:24', '2024-11-24 18:06:40', 'high'],
    ['240121', 'TC-Z2', '2024-11-24 18:11:14', '2024-11-24 18:20:06', 'low'],
    ['240121', 'OP-Z1', '2024-11-24 17:55:49', '2024-11-24 18:06:56', 'low'],
    ['240121', 'OP-Z2', '2024-11-24 17:57:38', '2024-11-24 18:10:16', 'low'],
    ['240133', 'TC-Z1', '2024-12-28 16:27:08', '2024-12-28 16:29:08', 'high'],
    ['240133', 'TC-Z1', '2024-12-28 16:35:24', '2024-12-28 16:38:18', 'low'],
    ['240133', 'TC-Z2', '2024-12-28 16:39:24', '2024-12-28 16:42:21', 'low'],
    ['240133', 'OP-Z1', '2024-12-28 16:26:02', '2024-12-28 16:31:16', 'low'],
    ['240133', 'OP-Z2', '2024-12-28 16:26:42', '2024-12-28 16:33:01', 'low'],
], columns=["LOT_NO", "SENSOR", "START", "END", "DIRECTION"])
review_ranges["START"] = pd.to_datetime(review_ranges["START"])
review_ranges["END"] = pd.to_datetime(review_ranges["END"])
review_ranges.insert(0, "REVIEW_ID", ["E" + str(i).zfill(3) for i in range(1, len(review_ranges) + 1)])
review_ranges["REASON"] = "여러 LOT에서 반복된 건조 온도 상승 후 하강"
review_ranges.loc[review_ranges["SENSOR"].str.startswith("OP-"), "REASON"] = "건조 온도 상승에 함께 낮아졌다 회복하는 출력"
print("확인한 센서 시간 범위:", len(review_ranges), "| LOT:", review_ranges["LOT_NO"].nunique())

### 4. 해당 센서의 후보만 해제하고 구역 라벨 다시 계산하기

- v3에 적힌 후보 센서에서 출발함. 새 후보를 추가하지 않음.
- 결측, 보류, 기존 제외 행은 그대로 둠.
- `TC-Z1` 후보를 해제해도 `OP-Z1` 후보가 남으면 Z1은 계속 `1`임.
- 해제 범위에 강종변경 또는 소둔로 정비 작업이 겹치면 해당 범위는 적용을 보류함.
  - 기록이 없다는 사실만으로 정상이라고 판단하지는 않음.
- v3의 Z3·Z6와 나머지 구역 라벨은 그대로 유지함.

In [ ]:
review_sensors = ["TC-Z1", "OP-Z1", "TC-Z2", "OP-Z2"]
before_flags = pd.DataFrame(False, index=base.index, columns=review_sensors)
for group in ["Z1", "Z2"]:
    text = ";" + base[group + "__LABEL_SENSORS"] + ";"
    for sensor in groups[group]:
        before_flags[sensor] = text.str.contains(";" + sensor + ";", regex=False) & base[group + "__LABEL"].eq("1")

removed = before_flags.copy() & False
review_ids = pd.DataFrame("", index=base.index, columns=["Z1", "Z2"])
range_rows = []
removed_rows = []
for row in review_ranges.itertuples(index=False):
    group = "Z1" if row.SENSOR.endswith("Z1") else "Z2"
    work = maintenance.loc[maintenance["STRT_DT"].lt(row.END) & maintenance["END_DT"].gt(row.START)]
    event = events.loc[events["EVT_DT"].ge(row.START) & events["EVT_DT"].lt(row.END)]
    in_range = base["LOT_NO"].eq(row.LOT_NO) & times.ge(row.START) & times.lt(row.END)
    center = lot_mean.loc[row.LOT_NO, row.SENSOR]
    same_direction = values[row.SENSOR].gt(center) if row.DIRECTION == "high" else values[row.SENSOR].lt(center)
    hit = in_range & not_excluded & before_flags[row.SENSOR] & same_direction
    apply_range = review_applied and work.empty and event.empty
    if not apply_range:
        hit = hit & False

    removed.loc[hit, row.SENSOR] = True
    review_ids.loc[hit, group] += row.REVIEW_ID + ";"
    detail = base.loc[hit, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    detail["GROUP"] = group
    detail["SENSOR"] = row.SENSOR
    detail["REVIEW_ID"] = row.REVIEW_ID
    removed_rows.append(detail)
    range_rows.append({
        "REVIEW_ID": row.REVIEW_ID,
        "REMOVED_SENSOR_ROWS": int(hit.sum()),
        "WORK_IDS": ";".join(work["WO_NO"].astype(str)),
        "GRADE_EVENT_IDS": ";".join(event["EVT_ID"].astype(str)),
        "APPLY_STATUS": "적용" if apply_range else "입력 변경 또는 조업 기록 겹침으로 보류",
    })

review_ranges = review_ranges.merge(pd.DataFrame(range_rows), on="REVIEW_ID", how="left")
removed_sensor_rows = pd.concat(removed_rows, ignore_index=True)
remaining_flags = before_flags & ~removed
label_change_parts = []
summary_rows = []
for group in ["Z1", "Z2"]:
    label_column = group + "__LABEL"
    old_label = base[label_column]
    touched = removed[groups[group]].any(axis=1)
    remains = remaining_flags[groups[group]].any(axis=1)
    changed = touched & old_label.eq("1") & ~remains

    output[group + "__LABEL_V3"] = old_label
    output.loc[changed, label_column] = "0"
    output.loc[changed, group + "__LABEL_STATUS"] = "후보 아님"
    output.loc[changed, group + "__LABEL_REASON"] = "확인한 건조 공통 움직임을 빼서 남은 후보 없음"
    output.loc[touched & remains, group + "__LABEL_REASON"] = "건조 공통 움직임을 뺐지만 같은 구역의 다른 후보가 남음"

    remaining_sensors = pd.Series("", index=base.index)
    for sensor in groups[group]:
        remaining_sensors.loc[remaining_flags[sensor]] += sensor + ";"
    output.loc[touched, group + "__LABEL_SENSORS"] = remaining_sensors.loc[touched].str.rstrip(";")
    output.loc[touched, group + "__COMMON_REVIEW_IDS"] = base.loc[touched, group + "__COMMON_REVIEW_IDS"] + review_ids.loc[touched, group]

    # 구역별 1→0뿐 아니라 센서 후보 일부만 해제되어 1이 남은 행도 기록함.
    detail = base.loc[touched, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    detail["GROUP"] = group
    detail["V3_LABEL"] = old_label.loc[touched]
    detail["V5_LABEL"] = output.loc[touched, label_column]
    detail["V3_SENSORS"] = base.loc[touched, group + "__LABEL_SENSORS"]
    detail["V5_SENSORS"] = output.loc[touched, group + "__LABEL_SENSORS"]
    detail["REVIEW_IDS"] = review_ids.loc[touched, group].str.rstrip(";")
    label_change_parts.append(detail)

    for lot in lots:
        in_lot = base["LOT_NO"].eq(lot)
        summary_rows.append({
            "LOT": lot, "구역": group,
            "v3 후보 1": int((in_lot & old_label.eq("1")).sum()),
            "v5 후보 1": int((in_lot & output[label_column].eq("1")).sum()),
            "1에서 0": int((in_lot & changed).sum()),
            "일부 센서 해제, 구역 1 유지": int((in_lot & touched & remains).sum()),
            "보류 또는 제외": int((in_lot & old_label.eq("")).sum()),
        })

label_changes = pd.concat(label_change_parts, ignore_index=True).sort_values(["LOT_NO", "MEAS_DT", "GROUP"])
summary = pd.DataFrame(summary_rows).sort_values(["LOT", "구역"])
output["DATASET_VERSION"] = "v5"
output["GROUP_LABEL_VERSION"] = "v3_plus_reviewed_drying_end_ranges_v5"
display(summary)
print("센서 후보 해제:", len(removed_sensor_rows), "| 구역 라벨 1→0:", int(summary["1에서 0"].sum()))
print("구역별 행수의 합임. 같은 원본 행에서 Z1과 Z2가 바뀌면 두 번 셈.")

### 5. 바뀐 라벨 표 바로 아래 같은 시간의 그래프 보기

- 표와 그래프 모두 실제 날짜와 시각을 사용함.
- 노란 배경: 확인한 해당 센서의 시간 범위.
- 초록 점: v3에서 후보였지만 이번에 해제한 센서값.
- 주황 점: v5에도 남은 센서 후보.
- 선은 원자료임. 센서값 자체는 바꾸지 않았음.
- `058`, `114`도 함께 보여주되 이번에는 라벨을 바꾸지 않음.

In [ ]:
range_names = {
    "SENSOR": "센서", "START": "시작, 포함", "END": "끝, 미포함",
    "DIRECTION": "방향", "REMOVED_SENSOR_ROWS": "해제한 센서 후보 수",
    "APPLY_STATUS": "적용 여부",
}
plot_sensors = ["TC-Z1", "OP-Z1", "TC-Z2", "OP-Z2"]
for lot in lots:
    print("LOT", lot, "| 마지막 기록", lot_end[lot])
    display(summary.loc[summary["LOT"].eq(lot)])
    ranges = review_ranges.loc[review_ranges["LOT_NO"].eq(lot)]
    if len(ranges):
        display(ranges[list(range_names)].rename(columns=range_names).replace({"high": "평균보다 위", "low": "평균보다 아래"}))
    else:
        print("온도와 출력 하강이 이어져 공통 움직임 해제를 적용하지 않음.")

    index = base.index[base["LOT_NO"].eq(lot) & times.ge(lot_end[lot] - pd.Timedelta(minutes=35))]
    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
    for axis, sensor in zip(axes.flat, plot_sensors):
        pieces = times.loc[index].diff().dt.total_seconds().gt(15).cumsum()
        for piece in pieces.unique():
            part = index[pieces.eq(piece)]
            axis.plot(times.loc[part], values.loc[part, sensor], color="#506f86", linewidth=0.7)
        selected = ranges.loc[ranges["SENSOR"].eq(sensor) & ranges["APPLY_STATUS"].eq("적용")]
        for row in selected.itertuples(index=False):
            axis.axvspan(row.START, row.END, color="#f4d368", alpha=0.25)
        dropped_index = index[removed.loc[index, sensor]]
        retained_index = index[remaining_flags.loc[index, sensor]]
        axis.scatter(times.loc[dropped_index], values.loc[dropped_index, sensor], s=4, color="#168b78", label="해제한 후보", zorder=3)
        axis.scatter(times.loc[retained_index], values.loc[retained_index, sensor], s=5, color="#dc7633", label="남은 후보", zorder=4)
        unit = "℃" if sensor.startswith("TC-") else "%"
        axis.set_title(sensor + " (" + unit + ")")
        axis.grid(alpha=0.15)
        axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        axis.set_xlim(lot_end[lot] - pd.Timedelta(minutes=35), lot_end[lot])
    for axis in axes[-1]:
        axis.set_xlabel("실제 시각")
    axes[0, 0].legend(loc="best", fontsize=9)
    grade = base.loc[index[0], "GRADE"]
    fig.suptitle("LOT " + lot + " | " + grade + " | " + lot_end[lot].strftime("%Y-%m-%d") + " 마지막 35분", fontsize=15)
    fig.tight_layout(rect=[0, 0.045, 1, 0.95])
    fig.text(0.06, 0.012, "노란 배경은 확인한 센서 시간. 초록 점만 후보 해제. 다른 센서 후보와 기존 제외는 유지함.", fontsize=10)
    if write_allowed:
        fig.savefig(os.path.join(FIG_DIR, "02_LOT_" + lot + "_label_changes.png"), dpi=130)
    plt.show()
    plt.close(fig)

### 6. v5와 변경 근거 저장하기

- `data/processed/cal01_base_v5.csv`: 기존 v3 행과 센서값, 수정한 구역 라벨.
- `review/v5/label_changes.csv`: 일부 센서의 후보가 해제된 행. `V3_LABEL`, `V5_LABEL`로 구역 라벨 변경 여부 확인.
- `review/v5/removed_sensor_rows.csv`: 후보를 해제한 정확한 센서와 원본 행.
- `review/v5/end_pattern_ranges.csv`: 적용한 센서, 시작, 끝, 방향과 이유.
- `Z1__LABEL_V3`, `Z2__LABEL_V3`는 비교용 이전 라벨임. 모델 입력에 넣지 않음.
- 모델을 다시 실행할 때는 `DATA_FILE = "cal01_base_v5.csv"`로 바꾸고 전체 실행함.
  - 이번 작업에서는 CSV를 만들었음. 모델 재학습 결과는 아직 없음.

In [ ]:
group_counts = []
for group in groups:
    label = output[group + "__LABEL"]
    status = output[group + "__LABEL_STATUS"]
    group_counts.append({
        "GROUP": group, "LABEL0": int(label.eq("0").sum()), "LABEL1": int(label.eq("1").sum()),
        "PENDING": int(status.eq("보류").sum()), "EXCLUDED": int(status.eq("제외").sum()),
    })

metadata = dict(source_metadata)
metadata["dataset_version"] = "v5"
metadata["master_file"] = OUTPUT_FILE
metadata["rows"] = len(output)
metadata["columns"] = len(output.columns)
metadata["group_counts"] = group_counts
metadata["issue"] = 106
metadata["source_sha256"] = {INPUT_FILE: source_sha256}
metadata["parent_source_sha256"] = source_metadata["source_sha256"]
for path in [SOURCE_META, EVENT_CSV, MAINT_CSV]:
    with open(path, "rb") as file:
        metadata["source_sha256"][os.path.relpath(path, ROOT)] = hashlib.file_digest(file, "sha256").hexdigest()
metadata["parent_label_policy"] = source_metadata["label_policy"]
metadata["label_policy"] = dict(source_metadata["label_policy"])
metadata["label_policy"]["version"] = "v3_plus_reviewed_drying_end_ranges_v5"
metadata["label_policy"]["v5_end_review"] = {
    "reviewed_source_sha256": REVIEWED_SHA256,
    "source_matches": bool(review_applied),
    "reviewed_lots": lots,
    "sensors": review_sensors,
    "range_count": len(review_ranges),
    "time_rule": "start inclusive, end exclusive; named sensor and direction only",
    "preserve_lots": ["240058", "240114"],
    "do_not_clear_entire_last_two_hours": True,
    "work_or_grade_event_overlap": "do not apply that range",
    "reviewed_after_observation_end": True,
    "normal_or_failure_confirmed": False,
}
metadata["review_scope"] = "14 LOT ends reviewed; drying sensor ranges added to unchanged v3 decisions"
metadata["model_input_forbidden"] = list(source_metadata["model_input_forbidden"]) + ["Z1__LABEL_V3", "Z2__LABEL_V3"]
metadata["end_review_summary"] = summary.to_dict("records")
metadata["models_trained"] = False

if write_allowed:
    output.to_csv(OUTPUT_CSV, index=False, encoding=ENC)
    review_ranges.to_csv(os.path.join(REVIEW_DIR, "end_pattern_ranges.csv"), index=False, encoding=ENC)
    removed_sensor_rows.to_csv(os.path.join(REVIEW_DIR, "removed_sensor_rows.csv"), index=False, encoding=ENC)
    label_changes.to_csv(os.path.join(REVIEW_DIR, "label_changes.csv"), index=False, encoding=ENC)
    summary.to_csv(os.path.join(REVIEW_DIR, "label_counts_before_after.csv"), index=False, encoding=ENC)
    lot_review.to_csv(os.path.join(REVIEW_DIR, "lot_end_review.csv"), index=False, encoding=ENC)
    pd.DataFrame(group_counts).to_csv(os.path.join(REVIEW_DIR, "group_label_counts.csv"), index=False, encoding=ENC)
    with open(OUTPUT_CSV, "rb") as file:
        metadata["master_sha256"] = hashlib.file_digest(file, "sha256").hexdigest()
    for name in [metadata["master_sha256"] + ".json", "dataset.json"]:
        with open(os.path.join(META_DIR, name), "w", encoding="utf-8") as file:
            json.dump(metadata, file, ensure_ascii=False, indent=2)
    print("저장:", OUTPUT_CSV)
    print("변경표:", REVIEW_DIR)
else:
    print("파일 저장 안 함. SAVE_FILES와 검토표 적용 여부를 확인하세요.")